In [ ]:
# =====================================================================
# STEP 1: Install Dependencies
# =====================================================================
!pip install sqlite-vss sentence-transformers pandas

import pandas as pd
import json
import sqlite3
import sqlite_vss
import re
from sentence_transformers import SentenceTransformer

# =====================================================================
# STEP 2: Initialize Embedding Model & Database Connection
# =====================================================================
print("Initializing MiniLM vector model (384 dimensions)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
VECTOR_DIMENSION = 384

db_path = "tmdb_movie_recommender.db"
conn = sqlite3.connect(db_path)

# Enable and load the SQLite Vector Similarity Search extension
conn.enable_load_extension(True)
sqlite_vss.load(conn)
conn.enable_load_extension(False)

cursor = conn.cursor()

# =====================================================================
# STEP 3: Define Relational and Vector Database Schema
# =====================================================================
print("Creating structural tables and vector tables...")

# 1. Main Movies Relational Metadata Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS movies (
    movie_id INTEGER PRIMARY KEY,
    title TEXT,
    overview TEXT,
    budget INTEGER,
    homepage TEXT,
    original_language TEXT,
    popularity REAL,
    release_date TEXT,
    revenue INTEGER,
    runtime REAL,
    status TEXT,
    tagline TEXT,
    vote_average REAL,
    vote_count INTEGER
);
""")

# 2. Relational Graph Attribute Tables
cursor.execute("CREATE TABLE IF NOT EXISTS movie_genres (movie_id INTEGER, genre_id INTEGER, genre_name TEXT);")
cursor.execute("CREATE TABLE IF NOT EXISTS movie_keywords (movie_id INTEGER, keyword_id INTEGER, keyword_name TEXT);")
cursor.execute("CREATE TABLE IF NOT EXISTS movie_cast (movie_id INTEGER, cast_id INTEGER, character TEXT, name TEXT, order_index INTEGER);")
cursor.execute("CREATE TABLE IF NOT EXISTS movie_crew (movie_id INTEGER, credit_id TEXT, department TEXT, job TEXT, name TEXT);")

# 3. Multi-Vector Virtual Table for Semantic Querying
cursor.execute(f"""
CREATE VIRTUAL TABLE IF NOT EXISTS vss_movie_vectors USING vss0(
    overview_embedding({VECTOR_DIMENSION}),
    metadata_text_embedding({VECTOR_DIMENSION})
);
""")
conn.commit()

# =====================================================================
# STEP 4: Bulletproof JSON String Parser
# =====================================================================
def clean_and_parse_json(val):
    """Safely converts TMDB escaped string format into structural Python objects."""
    if pd.isna(val) or not str(val).strip():
        return []
    try:
        # If it's already a clean string representation of a list, load it
        return json.loads(val)
    except json.JSONDecodeError:
        try:
            # Fix the escaped double-quote anomaly structure seen in raw preview
            cleaned = str(val).replace('""', '"')
            # Strip out wrapping quotes if the whole block got wrapped as a string literal
            if cleaned.startswith('"['): cleaned = cleaned[1:]
            if cleaned.endswith(']"'): cleaned = cleaned[:-1]
            return json.loads(cleaned)
        except:
            return []

# =====================================================================
# STEP 5: Read Datasets and Execute Conversion
# =====================================================================
print("Loading CSV files into pandas memory...")
df_movies = pd.read_csv('tmdb_5000_movies.csv')
df_credits = pd.read_csv('tmdb_5000_credits.csv')

# Sync keys and merge relational content cleanly
df_credits = df_credits.rename(columns={'movie_id': 'id'})
df_merged = pd.merge(df_movies, df_credits[['id', 'cast', 'crew']], on='id', how='inner')

print("Processing files and generating vector embeddings. This will take a moment...")

for idx, row in df_merged.iterrows():
    movie_id = int(row['id'])
    title = str(row['title_x'] if 'title_x' in row else row['title'])
    overview = str(row['overview']) if pd.notna(row['overview']) else ""

    # Parse structural text columns using our clean string decoder
    genres_list = clean_and_parse_json(row['genres'])
    keywords_list = clean_and_parse_json(row['keywords'])
    cast_list = clean_and_parse_json(row['cast'])
    crew_list = clean_and_parse_json(row['crew'])

    # 1. Insert structured movie data
    cursor.execute("""
    INSERT OR REPLACE INTO movies
    (movie_id, title, overview, budget, homepage, original_language, popularity, release_date, revenue, runtime, status, tagline, vote_average, vote_count)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        movie_id, title, overview, row['budget'], row['homepage'], row['original_language'],
        row['popularity'], row['release_date'], row['revenue'], row['runtime'],
        row['status'], row['tagline'], row['vote_average'], row['vote_count']
    ))

    # 2. Populate Graph Relationships
    genre_names = []
    for g in genres_list:
        cursor.execute("INSERT INTO movie_genres VALUES (?, ?, ?)", (movie_id, g.get('id'), g.get('name')))
        genre_names.append(str(g.get('name', '')))

    keyword_names = []
    for k in keywords_list:
        cursor.execute("INSERT INTO movie_keywords VALUES (?, ?, ?)", (movie_id, k.get('id'), k.get('name')))
        keyword_names.append(str(k.get('name', '')))

    for c in cast_list[:10]: # Store top 10 billed actors for optimal search weight
        cursor.execute("INSERT INTO movie_cast VALUES (?, ?, ?, ?, ?)", (movie_id, c.get('cast_id'), c.get('character'), c.get('name'), c.get('order')))

    for cr in crew_list:
        if cr.get('job') in ['Director', 'Screenplay', 'Writer']:
            cursor.execute("INSERT INTO movie_crew VALUES (?, ?, ?, ?, ?)", (movie_id, cr.get('credit_id'), cr.get('department'), cr.get('job'), cr.get('name')))

    # 3. Vector Feature Extraction
    # Vector 1: Semantic representation of the storyline
    overview_text = overview if overview.strip() else title
    overview_vector = model.encode(overview_text).tolist()

    # Vector 2: Hybrid contextual string for thematic/style matching
    combined_metadata_text = f"Title: {title}. Genres: {', '.join(genre_names)}. Keywords: {', '.join(keyword_names)}."
    metadata_vector = model.encode(combined_metadata_text).tolist()

    # 4. Insert directly into vector space binding with rowid
    cursor.execute(
        "INSERT INTO vss_movie_vectors(rowid, overview_embedding, metadata_text_embedding) VALUES (?, ?, ?)",
        (movie_id, json.dumps(overview_vector), json.dumps(metadata_vector))
    )

conn.commit()
print(f"\n[SUCCESS]: Relational Vector database successfully created at '{db_path}'!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.9 MB/s eta 0:00:00
Initializing MiniLM vector model (384 dimensions)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating structural tables and vector tables...
Loading CSV files into pandas memory...
Processing files and generating vector embeddings. This will take a moment...

[SUCCESS]: Relational Vector database successfully created at 'tmdb_movie_recommender.db'!


In [ ]:
import os

file_size = os.path.getsize('tmdb_movie_recommender.db')
print(f"File size: {file_size / (1024*1024):.2f} MB")

File size: 20.33 MB


In [ ]:
import csv

def inspect_file_delimiters(file_path):
    print(f"--- Analyzing Structure for: {file_path} ---")
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            # Read first line (headers) and second line (data sample)
            header_line = f.readline()
            data_line = f.readline()

            # Print raw characters to check for literal tabs, commas, or semicolons
            print("Raw Header Preview (First 150 chars):")
            print(repr(header_line[:150]))
            print("\nRaw Data Row Preview (First 150 chars):")
            print(repr(data_line[:150]))

            # Use Python's built-in sniffing tool to guess structural traits
            f.seek(0)
            sample = f.read(4096)
            dialect = csv.Sniffer().sniff(sample)
            print(f"\n[CONFIRMED DELIMITER]: '{dialect.delimiter}'")

    except Exception as e:
        print(f"Error inspecting file: {e}")
    print("\n" + "="*50 + "\n")

# Run inspections on your uploaded files
inspect_file_delimiters('tmdb_5000_movies.csv')
inspect_file_delimiters('tmdb_5000_credits.csv')


--- Analyzing Structure for: tmdb_5000_movies.csv ---
Raw Header Preview (First 150 chars):
'budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue'

Raw Data Row Preview (First 150 chars):
'237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""'

[CONFIRMED DELIMITER]: ','


--- Analyzing Structure for: tmdb_5000_credits.csv ---
Raw Header Preview (First 150 chars):
'movie_id,title,cast,crew\n'

Raw Data Row Preview (First 150 chars):
'19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id"": ""5602a8a7c3a3685532001c9a"", ""gender"": 2, ""id"": 65731, ""name"": '

[CONFIRMED DELIMITER]: ','


